In [ ]:
import joblib, uvicorn, asyncio, nest_asyncio, io, json, smtplib, csv, os
import pandas as pd
import numpy as np
from datetime import datetime
from email.message import EmailMessage
from fastapi import FastAPI, UploadFile, File, Request, BackgroundTasks
from fastapi.responses import HTMLResponse
from tensorflow.keras.models import load_model
LOG_FILE = "live_traffic_logs.csv"
nest_asyncio.apply()
folder = 'Elite_IDS_Final_v3'
SENDER_EMAIL = "........."#apni email dalein
APP_PASSWORD = "........" #app password dalein 
RECEIVER_EMAIL = "....."#apni email dalein
total_traffic_volume = 0   
total_security_events = 0  
app = FastAPI()
def save_dual_logs(attack_type, confidence, status, p_count=0, e_count=0):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = pd.DataFrame([{
        'timestamp': timestamp,
        'attack_type': attack_type,
        'confidence': confidence,
        'status': status,
        'packet_count': p_count,
        'event_count': e_count
    }])
    log_entry.to_csv('record.csv', mode='a', index=False, header=not os.path.exists('record.csv'))
    log_entry.to_csv('live.csv', mode='w', index=False, header=True)
def save_to_system_log(status, attack_type, confidence):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = f"[{timestamp}] Status: {status} | Type: {attack_type} | Conf: {confidence}%\n"
    with open("system_config.txt", "a") as f:
        f.write(log_entry)
try:
    rf_model = joblib.load(f'{folder}/rf_model.pkl')
    scaler_rf = joblib.load(f'{folder}/scaler_rf.pkl')
    autoencoder = load_model(f'{folder}/autoencoder_model.h5', compile=False) 
    scaler_dl = joblib.load(f'{folder}/scaler_dl.pkl')
    label_encoder = joblib.load(f'{folder}/ids_label_encoder.pkl')
    rf_features = list(scaler_rf.get_feature_names_out())
    print("🚀 SYSTEM v6.9: READY")
except Exception as e:
    print(f"🛑 Error: {e}")
def send_alert_email(attack_type, confidence, trigger):
    try:
        msg = EmailMessage()
        msg['Subject'] = f"🚨 SECURITY ALERT: {attack_type} Detected"
        msg['From'] = SENDER_EMAIL
        msg['To'] = RECEIVER_EMAIL
        content = f"ASIM ELITE IDS - REPORT\nType: {attack_type}\nConf: {confidence}%\nTrigger: {trigger}"
        msg.set_content(content)
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(SENDER_EMAIL, APP_PASSWORD)
            smtp.send_message(msg)
    except Exception as e:
        print(f"📧 Email Error: {e}")
HTML_CONTENT = """
<!DOCTYPE html>
<html>
<head>
    <title>ASIM ELITE IDS v6.9</title>
    <style>
        body { background: #0b0e14; color: #c9d1d9; font-family: 'Consolas', monospace; padding: 20px; display: flex; flex-direction: column; align-items: center; }
        .container { background: #161b22; padding: 30px; border-radius: 15px; border: 2px solid cyan; box-shadow: 0 0 40px rgba(0,255,255,0.15); max-width: 1000px; width: 100%; }
        h1 { color: cyan; text-align: center; text-shadow: 0 0 20px cyan; letter-spacing: 3px; }
        .card { background: rgba(255,255,255,0.02); padding: 20px; border-radius: 12px; margin-bottom: 20px; border: 1px solid #30363d; }
        .grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px; }
        input { background: #0d1117; border: 1px solid #30363d; color: cyan; padding: 12px; border-radius: 8px; width: 90%; }
        button { background: cyan; color: black; border: none; padding: 14px; border-radius: 8px; font-weight: 800; cursor: pointer; width: 100%; transition: 0.3s; }
        button:hover { transform: scale(1.02); box-shadow: 0 0 15px cyan; }
        #result { margin-top: 25px; padding: 25px; border-radius: 10px; display: none; border: 2px solid; transition: all 0.4s; }
        .alert { border-color: #ff4d4d; background: rgba(139, 0, 0, 0.6) !important; color: white; box-shadow: 0 0 30px rgba(255,77,77,0.4); }
        .normal { border-color: #2ea043; background: rgba(0, 100, 0, 0.4) !important; color: white; box-shadow: 0 0 30px rgba(46,160,67,0.3); }
        .batch-summary { display: flex; justify-content: space-around; padding: 20px; background: rgba(0,0,0,0.4); border-radius: 10px; margin-top: 10px; }
        .stat-box { text-align: center; }
        .stat-val { font-size: 1.8rem; font-weight: bold; display: block; }
        .badge { background: rgba(255,255,255,0.2); padding: 4px 10px; border-radius: 4px; font-size: 0.8rem; margin-right: 5px; }
    </style>
</head>
<body>
    <div class="container">
        <h1>🛡️ ASIM ELITE IDS v6.9</h1>
        <div class="card">
            <h3>⚡ REAL-TIME INSPECTION</h3>
            <div class="grid">
                <input type="number" id="port" placeholder="Dest Port">
                <input type="number" id="packets" placeholder="Fwd Packets">
                <input type="number" id="win" placeholder="Init Win Bytes">
                <input type="number" id="duration" placeholder="Flow Duration">
            </div>
            <button onclick="processStream()">SCAN PACKET</button>
        </div>
        <div class="grid">
            <div class="card"><h3>📄 JSON BATCH</h3><input type="file" id="jsonF"><button onclick="upload('json')">SCAN FILE</button></div>
            <div class="card"><h3>📊 CSV BATCH</h3><input type="file" id="csvF"><button onclick="upload('csv')" style="background:#2ea043; color:white;">SCAN FILE</button></div>
        </div>
        <div id="result"></div>
    </div>
    <script>
        function resetUI() {
            const res = document.getElementById('result');
            res.style.display = 'block'; res.className = '';
            res.innerHTML = "<h3 style='text-align:center;'>🌀 ANALYZING TRAFFIC...</h3>";
        }
        async function processStream(){
            resetUI();
            const d = {
                "Destination_Port": parseInt(document.getElementById('port').value) || 0,
                "Total_Fwd_Packets": parseInt(document.getElementById('packets').value) || 0,
                "Init_Win_bytes_forward": parseInt(document.getElementById('win').value) || 0,
                "Flow_Duration": parseInt(document.getElementById('duration').value) || 0
            };
            const r = await fetch('/predict', {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify(d)});
            show(await r.json());
        }
        async function upload(t){
            const f = document.getElementById(t+'F').files[0];
            if(!f) return alert("Select file!");
            resetUI();
            const d = new FormData(); d.append('file', f);
            const r = await fetch('/predict_file', {method:'POST', body:d});
            show(await r.json());
        }
        function show(data){
            const res = document.getElementById('result');
            if(data.type === "BATCH"){
                res.className = data.threats_found > 0 ? 'alert' : 'normal';
                res.innerHTML = `
                    <h2 style="text-align:center;">${data.verdict}</h2>
                    <div class="batch-summary">
                        <div class="stat-box"><span class="stat-val">${data.total_packets}</span><span>TOTAL</span></div>
                        <div class="stat-box"><span class="stat-val" style="color:#ff4d4d">${data.threats_found}</span><span>THREATS</span></div>
                        <div class="stat-box"><span class="stat-val" style="color:#2ea043">${data.total_packets - data.threats_found}</span><span>SAFE</span></div>
                    </div>
                `;
            } else {
                res.className = data.status === 'ALERT' ? 'alert' : 'normal';
                res.innerHTML = `
                    <h2 style="margin-top:0;">VERDICT: ${data.status}</h2>
                    <p><span class="badge">ATTACK TYPE</span> ${data.attack_type}</p>
                    <p><span class="badge">CONFIDENCE</span> ${data.rf_confidence}%</p>
                    <p><span class="badge">ANOMALY SCORE</span> ${data.anomaly_score}</p>
                    <p><span class="badge">TRIGGER</span> ${data.trigger}</p>
                `;
            }
        }
    </script>
</body>
</html>
"""
def sanitize_data(df):
    df.replace([np.inf, -np.inf], np.nan, inplace=True); df.fillna(0, inplace=True)
    return df.reindex(columns=rf_features, fill_value=0)
@app.get("/", response_class=HTMLResponse)
async def home(): return HTML_CONTENT
@app.post("/predict")
async def predict(request: Request, bg: BackgroundTasks):
    global total_traffic_volume, total_security_events
    raw = await request.json()
    if "custom_total_volume" in raw:
        total_traffic_volume = raw["custom_total_volume"]
        total_security_events = raw["custom_event_count"]
    else:
        total_security_events += 1
        fwd_pkts = raw.get("Total_Fwd_Packets", 0)
        total_traffic_volume = fwd_pkts 
    df_clean = sanitize_data(pd.DataFrame([raw]))
    probs = rf_model.predict_proba(scaler_rf.transform(df_clean))[0]
    classes = list(label_encoder.classes_); b_idx = classes.index("BENIGN")
    benign_p = probs[b_idx]
    mse = float(np.mean(np.power(scaler_dl.transform(df_clean) - autoencoder(scaler_dl.transform(df_clean), training=False), 2)))
    status, label, trigger = "NORMAL", "BENIGN (Safe)", "None"
    rf_conf = round(benign_p * 100, 2)
    if benign_p < 0.85:
        status, trigger = "ALERT", "Signature"
        a_idx = np.delete(np.arange(len(classes)), b_idx)
        label = classes[a_idx[np.argmax(probs[a_idx])]]
        rf_conf = round((1 - benign_p) * 100, 2)
    elif mse > 0.05:
        status, trigger, label = "ALERT", "Anomaly", "Unknown Exploit"
    fwd_pkts_val = raw.get("Total_Fwd_Packets", 0)
    flow_dur = raw.get("Flow_Duration", 0)
    duration_in_sec = flow_dur / 1000000 if flow_dur > 0 else 0.000001
    pps = fwd_pkts_val / duration_in_sec
    if pps > 20000 and fwd_pkts_val > 100: 
        status, trigger, label = "ALERT", "Heuristic (Rate-Limit)", "DDoS Flood"
    if status == "ALERT":
        bg.add_task(send_alert_email, label, rf_conf, trigger)
    save_to_system_log(status, label, rf_conf)
    save_dual_logs(label, rf_conf, status, total_traffic_volume, total_security_events) 
    return {"status": status, "attack_type": label, "rf_confidence": rf_conf, "anomaly_score": round(mse, 6), "trigger": trigger}
@app.post("/predict_file")
async def predict_file(file: UploadFile = File(...)):
    global total_traffic_volume, total_security_events
    contents = await file.read()
    df = pd.read_csv(io.BytesIO(contents)) if file.filename.endswith('.csv') else pd.DataFrame(json.loads(contents))
    total_traffic_volume += len(df)
    total_security_events += 1
    df_clean = sanitize_data(df)
    is_attack = rf_model.predict_proba(scaler_rf.transform(df_clean))[:, list(label_encoder.classes_).index("BENIGN")] < 0.85
    mses = np.mean(np.power(scaler_dl.transform(df_clean) - autoencoder.predict(scaler_dl.transform(df_clean), batch_size=1024, verbose=0), 2), axis=1)
    total_threats = int(np.sum(is_attack | (mses > 0.04)))
    verdict = "BREACH DETECTED" if total_threats > 0 else "SYSTEM SECURE"
    save_dual_logs(verdict, "BATCH", "BATCH", total_traffic_volume, total_security_events)
    return {"type": "BATCH", "total_packets": len(df), "threats_found": total_threats, "verdict": verdict}
if __name__ == "__main__":
    config = uvicorn.Config(app, host="127.0.0.1", port=8000)
    server = uvicorn.Server(config)
    loop = asyncio.get_event_loop()
    if loop.is_running(): loop.create_task(server.serve())
    else: loop.run_until_complete(server.serve())

🚀 SYSTEM v6.9: READY


INFO:     Started server process [5668]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:64404 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:64404 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:52784 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:50368 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:50425 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:51191 - "POST /predict_file HTTP/1.1" 200 OK
INFO:     127.0.0.1:54644 - "POST /predict_file HTTP/1.1" 200 OK
INFO:     127.0.0.1:65074 - "POST /predict_file HTTP/1.1" 200 OK
INFO:     127.0.0.1:54939 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:54940 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:54941 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:54942 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:54943 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:54944 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:54945 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:54946 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:549